# Playground for CircuitCollector


## Test DeepSeek API


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="sk-76c98016d6194644a03400a9d2bea41a", base_url="https://api.deepseek.com"
)


system_prompt = (
    "你是一个 gm/Id 方法驱动的模拟电路设计代理。你必须通过工具调用获得数值结果（仿真与 OP 回读），不得主观猜数值。"
    "优先满足硬约束（饱和余量、稳定性PM/UGF、输出摆幅、功耗上限），再优化软指标（增益、噪声、面积）。"
    "每次行动只微调“角色一致”的少量参数（如：输入对一起调 W/M；补偿 C 单独加；二级电流成对调整）。"
    "输出采用三段式："
    "1) COT-brief：一句话诊断+策略，不展开细节推理；"
    "2) ACTION：一个函数调用（run_sim），其中仅包含改动后的参数；"
    "3) EXPECTED：定性预期（如 PM上升、UGF下降少量、功耗上升小幅）。"
)
dev_prompt = (
    "拓扑角色：NMOS 差分对(XM3/XM4)、尾源(XM6)、PMOS 镜像/负载(XM1/XM2)、二级 PMOS 放大(XM7)+NMOS 下拉(XM8)、偏置(XM5+Ib)、米勒补偿(XC1)。"
    "目标分解：UGF≈gm1/(2πCc)；SR≈I2/Cc；增益≈∏(gm·ro)；饱和余量 Δ≈50–100 mV。"
    "动作优先级：修硬约束>调补偿>调电流分配>微调尺寸；避免来回震荡；若方向不明，先小步增/减 Cc 或输入对 gm，再观察。"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": dev_prompt},
        {"role": "user", "content": ""},
    ],
    stream=False,
)

print(response.choices[0].message.content)

In [10]:
# file: deepseek_gmid_agent_demo.py
from __future__ import annotations
import json
import os
from typing import Dict, Any, List, Tuple
from openai import OpenAI
from sim_api import SimulationAPI


# ========= 1) DeepSeek 客户端 =========
def make_client(api_key: str | None = None) -> OpenAI:
    api_key = api_key or os.getenv("DEEPSEEK_API_KEY") or ""
    if not api_key:
        raise RuntimeError("Missing DEEPSEEK_API_KEY")
    return OpenAI(api_key=api_key, base_url="https://api.deepseek.com")


# ========= 2) 工具实现：run_sim（归一化/配对/步长/审查/自适应） =========
api = SimulationAPI()

# 步长限制（每步最大改动），防止 LLM 一次迈太大步
BASE_MAX_STEP = {
    "_L": 0.15,  # µm
    "_W": 2.0,  # µm
    "_M": 4,  # fingers
    "C1_L": 6,  # 整数格
    "C1_W": 6,  # 整数格
    "ibias": 2e-5,  # A
}

# 允许配对的器件：输入对 & 二级
PAIR_GROUPS = [("M3", "M4"), ("M7", "M8")]

# 记录最近一次成功应用的参数 & 最近一次 specs/op（用于护栏/自适应）
_last_params_applied: Dict[str, Any] | None = None
_last_op_snapshot: Dict[str, Any] | None = None  # op_region
_last_specs_snapshot: Dict[str, Any] | None = None  # specs
_sat_bad_streak: int = 0  # 连续欠饱和轮数
INIT_PARAMS: Dict[str, Any] = {}  # 最小尺寸基准


# ======== 工具函数 ========
def _to_um(val) -> float:
    x = float(val)
    return x * 1e6 if abs(x) < 1e-3 else x


def _to_cap_step(val) -> int:
    return int(round(float(val)))


def _parse_rel(val) -> Tuple[bool, float]:
    """
    支持把值写成字符串 '+2' / '-0.05' 表示相对变更。
    返回 (is_delta, numeric_value)
    """
    if isinstance(val, str) and val.strip() and val.strip()[0] in "+-":
        try:
            return True, float(val)
        except Exception:
            pass
    return False, float(val)


def _normalize_params(raw: Dict[str, Any]) -> Dict[str, Any]:
    """
    归一化到大写扁平键（单位：µm；M为整数）。
      {"m6":{"m":3,"l":0.25e-6}} -> {"M6_M":3,"M6_L":0.25}
      {"C1_W":20} -> 保留
      {"m7_m":"+2"} -> 视为相对（delta）
    额外：支持字符串前缀 + / - 表示 delta（由上层合并时处理）
    """
    out: Dict[str, Any] = {}

    # 已扁平
    for k, v in raw.items():
        if isinstance(k, str) and (
            (k.upper().startswith("M") and k[-2:] in ("_L", "_W"))
            or k.upper().endswith("_M")
        ):
            out[k.upper()] = v
        elif k in ("C1_L", "C1_W", "ibias"):
            out[k] = v

    # 嵌套/小写
    for k, v in raw.items():
        if not isinstance(k, str):
            continue
        key = k.lower()
        if key.startswith("m") and key[1:].isdigit():
            idx = key[1:]
            if isinstance(v, dict):
                if "l" in v:
                    out[f"M{idx}_L"] = _to_um(v["l"])
                if "w" in v:
                    out[f"M{idx}_W"] = _to_um(v["w"])
                if "m" in v:
                    out[f"M{idx}_M"] = int(v["m"])
            else:
                parts = key.split("_", 1)
                if len(parts) == 2:
                    suf = parts[1]
                    if suf == "l":
                        out[f"M{idx}_L"] = _to_um(v)
                    if suf == "w":
                        out[f"M{idx}_W"] = _to_um(v)
                    if suf == "m":
                        out[f"M{idx}_M"] = int(v)
        elif key == "c1":
            if isinstance(v, dict):
                if "l" in v:
                    out["C1_L"] = _to_cap_step(v["l"])
                if "w" in v:
                    out["C1_W"] = _to_cap_step(v["w"])
        elif key == "c1_l":
            out["C1_L"] = _to_cap_step(v)
        elif key == "c1_w":
            out["C1_W"] = _to_cap_step(v)
        elif key == "ibias":
            out["ibias"] = float(v)

    return out


def _auto_pair(params: Dict[str, Any]) -> Dict[str, Any]:
    out = params.copy()
    for a, b in PAIR_GROUPS:
        for suf in ("_L", "_W", "_M"):
            ka, kb = f"{a}{suf}", f"{b}{suf}"
            if ka in params and kb not in params:
                out[kb] = params[ka]
            if kb in params and ka not in params:
                out[ka] = params[kb]
    return out


def _effective_max_step() -> Dict[str, Any]:
    # 自适应步长：sat_margin 连续为负时放宽 M5_W/M6_L
    global _sat_bad_streak
    ms = BASE_MAX_STEP.copy()
    if _last_op_snapshot:
        sat = _last_op_snapshot.get("m6", {}).get("sat_margin", None)
        if sat is not None and sat < 0:
            _sat_bad_streak += 1
        else:
            _sat_bad_streak = 0
    if _sat_bad_streak >= 2:
        ms["_W"] = BASE_MAX_STEP["_W"] * 1.5
        ms["_L"] = BASE_MAX_STEP["_L"] * 1.5
    return ms


def _apply_step_limit(
    params: Dict[str, Any], last_params: Dict[str, Any]
) -> Dict[str, Any]:
    """将相对于 last_params 的改动限制在 MAX_STEP 内。"""
    lims = _effective_max_step()
    out = {}
    for k, v in params.items():
        base = last_params.get(k)
        if base is None:
            out[k] = v
            continue
        # 支持字符串相对语义
        is_delta, num = _parse_rel(v)
        target = base + num if is_delta else num
        dv = target - base
        if k in ("C1_L", "C1_W"):
            cap_lim = lims[k]
            dv = max(-cap_lim, min(cap_lim, dv))
            out[k] = int(round(base + dv))
        elif k.endswith("_M"):
            lim = lims["_M"]
            dv = max(-lim, min(lim, dv))
            out[k] = int(round(base + dv))
        elif k.endswith("_L"):
            lim = lims["_L"]
            dv = max(-lim, min(lim, dv))
            out[k] = round(base + dv, 3)
        elif k.endswith("_W"):
            lim = lims["_W"]
            dv = max(-lim, min(lim, dv))
            out[k] = round(base + dv, 3)
        elif k == "ibias":
            lim = lims["ibias"]
            dv = max(-lim, min(lim, dv))
            out[k] = base + dv
        else:
            out[k] = target
    return out


def _min_size_hard_check(proposed: Dict[str, Any], init_params: Dict[str, Any]):
    errs = []
    for k, v in proposed.items():
        if k.endswith(("_L", "_W", "_M")) and k in init_params:
            if v < init_params[k]:
                errs.append(f"{k}={v} < init {init_params[k]}")
    if errs:
        raise ValueError("ActionRejected(min-size): " + " | ".join(errs))


def _limit_param_count(proposed: Dict[str, Any]):
    # 约束每次 ≤4 项（按最终提交键计数）
    if len(proposed.keys()) > 4:
        raise ValueError(f"ActionRejected(param-count): got {len(proposed)} > 4")


def _phase_guard(proposed, last_params, last_specs, last_op):
    pm = (last_specs or {}).get("phase_margin")
    sat = (last_op or {}).get("m6", {}).get("sat_margin")

    # Phase A：sat<0.05 或 pm<60°
    if (sat is not None and sat < 0.05) or (pm is not None and pm < 60):
        forbid = ("M3_", "M4_")  # 禁止动 M3/M4
        if any(k.startswith(forbid_i) for k in proposed for forbid_i in forbid):
            raise ValueError("ActionRejected(A): fix tail/PM first")
        # 方向性限制：M5_W 只能↑，ibias 只能↓，M6_L 只能↑
        if "M5_W" in proposed and proposed["M5_W"] < last_params.get("M5_W", 0):
            raise ValueError("ActionRejected(A): M5_W must not decrease")
        if "ibias" in proposed and proposed["ibias"] > last_params.get("ibias", 0):
            raise ValueError("ActionRejected(A): ibias must not increase")
        if "M6_L" in proposed and proposed["M6_L"] < last_params.get("M6_L", 0):
            raise ValueError("ActionRejected(A): M6_L must not decrease")

    # Phase B：60°≤PM<65°
    if pm is not None and 60 <= pm < 65:
        # 仍禁止 M3/M4；C1 的步长≤4
        if any(k.startswith(("M3_", "M4_")) for k in proposed):
            raise ValueError("ActionRejected(B): avoid M3/M4; tune C1 or M7/M8_M")
        for capk in ("C1_W", "C1_L"):
            if capk in proposed and abs(proposed[capk] - last_params.get(capk, 0)) > 4:
                raise ValueError("ActionRejected(B): C1 step too large")

    # Phase C：sat≥0.05 且 PM≥62°
    if (sat is not None and sat >= 0.05) and (pm is not None and pm >= 62):
        # 禁止通过 W↑ 拉增益；L 单步≤0.10µm
        if any(k.endswith("_W") for k in proposed):
            raise ValueError("ActionRejected(C): avoid W↑; use L↑ and mild M↑")
        for k in ("M3_L", "M4_L", "M7_L", "M8_L"):
            if k in proposed and (proposed[k] - last_params.get(k, 0)) > 0.10:
                raise ValueError("ActionRejected(C): L step too large")


def _guardrails(proposed, last_params, last_specs, last_op):
    """软性护栏：给出更细的错误信息（不重复 _phase_guard 的硬拦）"""
    errs = []
    pm = (last_specs or {}).get("phase_margin")
    sat = (last_op or {}).get("m6", {}).get("sat_margin")

    # A阶段：sat<0.05V 或 pm<60° -> 只准“救偏置/保相位”的动作
    if (sat is not None and sat < 0.05) or (pm is not None and pm < 60):
        if any(k in proposed for k in ("M3_M", "M3_W", "M4_M", "M4_W")):
            errs.append("Phase-A: forbid changing M3/M4 sizing until tail/PM fixed")
        if "M5_W" in proposed and proposed["M5_W"] < last_params.get("M5_W", 0):
            errs.append("Phase-A: M5_W must not decrease")
        if "ibias" in proposed and proposed["ibias"] > last_params.get("ibias", 0):
            errs.append("Phase-A: ibias must not increase")
        if "M6_L" in proposed and proposed["M6_L"] < last_params.get("M6_L", 0):
            errs.append("Phase-A: M6_L must not decrease")

    # B阶段：60°≤PM<65°
    if pm is not None and 60 <= pm < 65:
        if any(k in proposed for k in ("M3_M", "M3_W", "M4_M", "M4_W")):
            errs.append("Phase-B: avoid M3/M4 sizing; tune C1 or M7/M8_M instead")
        for capk in ("C1_W", "C1_L"):
            if capk in proposed and abs(proposed[capk] - last_params.get(capk, 0)) > 4:
                errs.append("Phase-B: C1 step too large (>4)")

    # C阶段：当 sat≥0.05 且 PM≥62° 才允许拉增益；优先 L↑（ro↑），禁止靠 W↑
    if sat is not None and sat >= 0.05 and pm is not None and pm >= 62:
        if any(k in proposed for k in ("M3_W", "M4_W", "M7_W", "M8_W")):
            errs.append("Phase-C: avoid W↑ for gain; use L↑ and mild M↑")
        for k in ("M3_L", "M4_L", "M7_L", "M8_L"):
            if k in proposed and (proposed[k] - last_params.get(k, 0)) > 0.1:
                errs.append("Phase-C: L step too large (>0.1µm)")

    if errs:
        raise ValueError("ActionRejected(guardrails): " + " | ".join(errs))


def _pretty_diff_log(
    last_params: Dict[str, Any], target: Dict[str, Any], final_: Dict[str, Any]
):
    print("---- param change log (base → target(delta) → final) ----")
    keys = sorted(set(target.keys()) | set(final_.keys()))
    for k in keys:
        base = last_params.get(k, None)
        t = target.get(k, None)
        f = final_.get(k, None)
        is_delta, _ = _parse_rel(t) if t is not None else (False, None)
        tag = "(Δ)" if is_delta else ""
        print(f"{k:>8s}: base={base}  target{tag}={t}  final={f}")
    print("---------------------------------------------------------")


def _merge_relative(last_params: Dict[str, Any], raw: Dict[str, Any]) -> Dict[str, Any]:
    """
    把 raw（可能含相对字符串）转换成**目标值**（target）。
    注意：步长限制在下一步做。
    """
    out = {}
    for k, v in raw.items():
        base = last_params.get(k)
        is_delta, num = _parse_rel(v)
        out[k] = (base + num) if (base is not None and is_delta) else num
    return out


# ========= 3) 工具入口 =========
def tool_run_sim(kwargs: Dict[str, Any]) -> Dict[str, Any]:
    """
    kwargs: {"params": {...}, "mode": "delta" | "absolute"} 或 {"delta": {...}}
    自动：归一化 → 配对镜像 → 解析相对 → 步长裁剪 → 最小尺寸硬检查 → 阶段护栏 → 软护栏 → 仿真。
    """
    global _last_params_applied, _last_op_snapshot, _last_specs_snapshot

    if _last_params_applied is None:
        raise RuntimeError(
            "tool_run_sim called before _last_params_applied is set by main()."
        )

    raw_params = kwargs.get("params") or {}
    mode = kwargs.get("mode", "absolute")
    if "delta" in kwargs and isinstance(kwargs["delta"], dict):
        # 允许把相对修改放在 delta 字段
        raw_params = {**raw_params, **kwargs["delta"]}
        mode = "delta"

    flat = _normalize_params(raw_params)
    paired = _auto_pair(flat)

    # 解析相对：两种方式都支持
    if mode == "delta":
        target = _merge_relative(_last_params_applied, paired)
    else:
        # absolute，但是对字符串 "+/-" 的键仍视作相对
        target = {}
        for k, v in paired.items():
            base = _last_params_applied.get(k)
            is_delta, num = _parse_rel(v)
            target[k] = (base + num) if (base is not None and is_delta) else num

    limited = _apply_step_limit(target, _last_params_applied)

    # 每次 ≤4 项（按最终提交键计数）
    _limit_param_count(limited)

    # 最小尺寸硬约束
    _min_size_hard_check(limited, INIT_PARAMS)

    # 阶段硬护栏（基于上一轮 specs/op）
    _phase_guard(
        limited,
        _last_params_applied,
        _last_specs_snapshot or {},
        _last_op_snapshot or {},
    )

    # 软护栏（信息性错误）
    _guardrails(
        limited,
        _last_params_applied,
        _last_specs_snapshot or {},
        _last_op_snapshot or {},
    )

    # 打印四元组日志
    print("[tool_run_sim] raw   :", raw_params)
    print("[tool_run_sim] flat  :", flat)
    print("[tool_run_sim] paired:", paired)
    _pretty_diff_log(_last_params_applied, target, limited)

    # 合并并仿真
    merged = _last_params_applied.copy()
    merged.update(limited)

    out = api.run(params=merged)
    _last_params_applied = merged
    _last_op_snapshot = out.get("op_region", {})
    _last_specs_snapshot = out.get("specs", {})

    return {"specs": out["specs"], "op": out["op_region"], "applied_params": merged}


# ========= 4) 提示词（促使“成组行动+多器件观察”） =========
SYSTEM_PROMPT = (
    "你是一个 gm/Id 方法驱动的模拟电路设计代理。"
    "必须通过调用工具 run_sim 获取真实仿真结果（specs 与 OP），不得臆测数值。"
    "每次行动请**成组微调（最多4项）**，并**同时观察首级差分对 (M3/M4)、二级跨导对 (M7/M8)、尾源 (M6)** 的 gm/Id、VDS−Vdsat，"
    "以及与补偿电容 Cc 的关系（UGF≈gm1/(2πCc)，fz≈gm2/(2πCc)）。"
    "硬约束优先：M6 饱和 (VDS−Vdsat ≥ 0.05~0.10 V)、PM ≥ 60°；再优化 dcgain、GBW 与功耗。"
    "参数键与单位：Mx_L/Mx_W（µm）、Mx_M（整数）、C1_L/C1_W（整数步）、ibias（安培）。"
    "目标是 dcgain > 60，PM > 60°，GBW > 2 MHz；若冲突，优先保证 dcgain。"
    "晶体管的尺寸不得小于初始值。"
    "优先用并指数 M 调 gm/UGF/PM；W 仅用于降 Vov6（如 M6），L 主要用于拉 ro/增益；在 A 阶段禁止靠 W↑追增益。"
    "输出必须分三段：\n"
    "1) COT-brief：一句话诊断 + 策略；\n"
    "2) ACTION：调用 run_sim，提供一组协调的小步参数修改（≤4项）；\n"
    "3) EXPECTED：对 PM/UGF/功耗/增益的定性预期。"
    "注意：若你的 ACTION 违反工具返回的错误提示（如最小尺寸/欠饱和护栏），你必须改换动作再试。"
    "默认将未带 ± 号的数值按 absolute 解释；如含 ± 号或 mode=delta，则按相对变更执行。"
)

DEV_PROMPT = (
    "拓扑：M3/M4 组成首级差分 (gm1)，M7/M8 为二级 (gm2)，M6 尾源，C1 为米勒补偿。"
    "规则：\n"
    "- 首选 M： 调 gm 先 M3/M4_M、M7/M8_M ↑，再考虑小幅 W；\n"
    "- 若 PM 低但 GBW 尚可，多因 RHP 零点：先增 gm2 (M7/M8_M ↑)，必要时小加 Cc；\n"
    "- 若 GBW 低：先增 gm1 (M3/M4_M ↑)，或小降 Cc；\n"
    "- 当 M6 欠饱和 (VDS−Vdsat<0)：先通过 **M5_W↑ 或 ibias↓** 降低 Vov6，并可小幅 **M6_L↑**；"
    "  禁止提出会进一步拉低 net2 的动作（如 M5_W↓、ibias↑、显著放大 M3/M4 的 W·M）；\n"
    "- 参数成对：M3 与 M4、M7 与 M8 的 L/W/M 必须同步；\n"
    "- 每次行动最多提出 4 个协调调整；\n"
    "- 所有数值以工具返回为准，禁止臆测/伪造读数；\n"
    "策略：先找到满足“可行化（M6 饱和 & PM≥60）”的工作点，再在局部用少量步数拉升 dcgain/GBW。"
    "工作流阶段：分三步，只有阶段目标满足后才进入下一阶段；每次行动≤4项。"
    "阶段A【可行化】：若 M6 饱和裕度<0.05V 或 PM<60°，只允许：M5_W↑、ibias↓、M6_L↑、(必要时) C1_W/L↑、M7/M8_M↑；禁止动 M3/M4。"
    "阶段B【相位稳固】：当 M6 饱和且 PM∈[60°,65°) 时，仅允许 C1_W/L 与 M7/M8_M 的微调把 PM 锁到≥62°；禁止动 M3/M4。"
    "阶段C【增益拉升】：仅在 A/B 达标后允许动 M3/M4/M7/M8 的 L（优先 L↑ 提 ro）；每次 L≤+0.10µm，M 仅小幅补偿极点；禁止通过 W↑ 追增益。"
    "增益优先级（阶段C生效）：(1) M3/M4_L +0.05~0.10µm → ro1↑；(2) M7/M8_L +0.05µm → ro2↑；(3) 若 GBW 下滑明显，则 M3/M4_M +1~2 抵消 UGF 下跌；(4) 视情况 ibias 小降(≥-2e-6) 提高 gm/Id，需用 M7/M8_M 微补 gm2；(5) 禁止 W↑。"
)

# ========= 5) 工具 Schema =========
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_sim",
            "description": "运行电路仿真。键：Mx_L/Mx_W/Mx_M, C1_L/C1_W, ibias；支持嵌套/小写；自动配对 M3/M4、M7/M8；支持相对模式。",
            "parameters": {
                "type": "object",
                "properties": {
                    "params": {
                        "type": "object",
                        "description": '成组微调（≤4项），如 {"m6":{"l":"+0.05e-6"}, "m5":{"w":"+0.1"}, "c1_w":"+2"}',
                        "additionalProperties": True,
                    },
                    "mode": {
                        "type": "string",
                        "enum": ["absolute", "delta"],
                        "description": "参数含义：absolute=绝对值；delta=相对变更（推荐）。",
                    },
                    "delta": {
                        "type": "object",
                        "description": "可选：把相对变更单独放到这里（与 mode=delta 等价）。",
                    },
                },
                "required": ["params"],
                "additionalProperties": False,
            },
        },
    }
]


# ========= 6) 主循环 =========
def main():
    # 建议用环境变量：export DEEPSEEK_API_KEY=...
    client = make_client(api_key="sk-76c98016d6194644a03400a9d2bea41a")

    # 初始参数（作为最小尺寸的基准）
    init = {
        "M1_L": 0.15,
        "M1_W": 0.55,
        "M1_M": 1,
        "M3_L": 0.15,
        "M3_W": 0.42,
        "M3_M": 1,
        "M5_L": 0.15,
        "M5_W": 0.42,
        "M5_M": 1,
        "M6_L": 0.15,
        "M6_W": 0.42,
        "M6_M": 1,
        "M7_L": 0.15,
        "M7_W": 0.55,
        "M7_M": 1,
        "M8_L": 0.15,
        "M8_W": 0.42,
        "M8_M": 1,
        "C1_L": 18,
        "C1_W": 18,
        "ibias": 3e-5,
    }

    global _last_params_applied, _last_op_snapshot, _last_specs_snapshot, INIT_PARAMS
    INIT_PARAMS = init.copy()

    # 先跑一次仿真，并设置 last_params/specs/op
    _last_params_applied = init.copy()
    first = api.run(params=init)
    _last_op_snapshot = first.get("op_region", {})
    _last_specs_snapshot = first.get("specs", {})

    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "system", "content": DEV_PROMPT},
        {
            "role": "user",
            "content": "这是初始仿真读数（含 OP），请按 gm/Id 规则提出**成组微调**并通过 ACTION 调用 run_sim（推荐 mode=delta）：\n"
            + json.dumps(
                {
                    "specs": first["specs"],
                    "op": first["op_region"],
                    "applied_params": init,
                },
                ensure_ascii=False,
                indent=2,
            ),
        },
    ]

    for turn in range(20):
        # 让模型给出成组 ACTION（启用工具调用）
        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            tools=TOOLS,
            tool_choice="required",
            temperature=0.2,
        )
        msg = resp.choices[0].message

        assistant_msg = {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in (msg.tool_calls or [])
            ],
        }
        messages.append(assistant_msg)

        if not getattr(msg, "tool_calls", None):
            print("\n[LLM OUTPUT]\n", msg.content)
            break

        for tc in msg.tool_calls:
            if tc.function.name != "run_sim":
                continue
            args = json.loads(tc.function.arguments or "{}")
            print(
                "\n[ACTION → run_sim args]\n",
                json.dumps(args, ensure_ascii=False, indent=2),
            )
            try:
                tool_result = tool_run_sim(args)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "name": "run_sim",
                        "content": json.dumps(tool_result, ensure_ascii=False),
                    }
                )
            except Exception as e:
                # 把结构化错误抛回去，迫使 LLM 改动作
                err_payload = {"error": str(e)}
                print("[run_sim ERROR]", err_payload)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "name": "run_sim",
                        "content": json.dumps(err_payload, ensure_ascii=False),
                    }
                )

        # summary
        resp2 = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=0.2,
        )
        msg2 = resp2.choices[0].message
        print("\n[LLM OUTPUT]\n", msg2.content)
        messages.append({"role": "assistant", "content": msg2.content})

    print("\n✓ Demo done.")


if __name__ == "__main__":
    main()

Simulation strategy: Use parameters from circuit.params section
Running single simulation...
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/temp/test_opamp_params.txt
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/opamp1_circuit.cir
Generated testbench circuit: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/opamp1_circuit.cir
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_op_region.spice
Generated circuit op region: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_op_region.spice
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_circuit_params.txt
Generated circuit parameters: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_circuit_params.txt
[⚡] Running simulation: opamp1_circuit.cir
[✓] Simulation completed: /home/G30079315/code/C

In [ ]:
import os
from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

os.environ["DEEPSEEK_API_KEY"] = "sk-76c98016d6194644a03400a9d2bea41a"


model = LitellmModel(
    model="deepseek/deepseek-chat", api_key=os.environ["DEEPSEEK_API_KEY"]
)

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant",
    model=model,
)
set_tracing_disabled(disabled=True)

result = await Runner.run(
    agent, "Write a sample code for a simple web server in Python."
)
print(result.final_output)

Here's a simple web server implementation in Python using the built-in `http.server` module:

## Basic HTTP Server

```python
import http.server
import socketserver

PORT = 8000

# Create a simple HTTP request handler
class SimpleHTTPRequestHandler(http.server.SimpleHTTPRequestHandler):
    def do_GET(self):
        # Set response headers
        self.send_response(200)
        self.send_header('Content-type', 'text/html')
        self.end_headers()
        
        # Send response body
        response = f"""
        <html>
        <head><title>Simple Python Server</title></head>
        <body>
            <h1>Hello from Python Web Server!</h1>
            <p>You requested: {self.path}</p>
            <p>Server is running on port {PORT}</p>
        </body>
        </html>
        """
        self.wfile.write(response.encode('utf-8'))

# Start the server
with socketserver.TCPServer(("", PORT), SimpleHTTPRequestHandler) as httpd:
    print(f"Server running at http://localhost:{PORT}")
 

## Test Sim API


In [ ]:
# CircuitCollector/test/test_simulation_runner_api_demo.py
from sim_api import SimulationAPI

if __name__ == "__main__":
    api = SimulationAPI()
    out = api.run(params={"M1_L": 0.15, "M1_W": 0.55, "M1_M": 1})
    print("Specs:", out["specs"])
    print("OP  :", out["op_region"])

Simulation strategy: Use parameters from circuit.params section
Running single simulation...
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/temp/test_opamp_params.txt
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/opamp1_circuit.cir
Generated testbench circuit: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/opamp1_circuit.cir
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_op_region.spice
Generated circuit op region: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_op_region.spice
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_circuit_params.txt
Generated circuit parameters: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_circuit_params.txt
[⚡] Running simulation: opamp1_circuit.cir
[✓] Simulation completed: /home/G30079315/code/C

## Active Learning for Data Collection


In [1]:
# file: al_gmid_demo.py
from __future__ import annotations
import random, math, csv, os, time
from typing import Dict, Any, List, Tuple
from dataclasses import dataclass
from statistics import mean
from sim_api import SimulationAPI

# ============ 目标与硬约束 ============
TARGETS = {
    "dcgain_": 50.0,  # dB
    "gain_bandwidth_product_": 2e6,  # Hz
    "phase_margin": 60.0,  # deg
}
HARD_CONSTRAINTS = {
    "pm_min": 55.0,  # 初期放宽，AL 过程中逼到 60
    "m6_sat_margin_min": 0.05,  # V, M6: vds - vdsat
}
# gm/Id 合理带（经验区间），单位 1/V
GMID_WINDOW = {
    "m3": (5.0, 25.0),  # 首级差分
    "m4": (5.0, 25.0),
    "m7": (5.0, 20.0),  # 二级
    "m8": (5.0, 20.0),
}

# ============ 搜索空间（尽量收敛在 M，小步动 L/W） ============
# 以你的 PDK/配置为准：L/W 单位 µm；M 整数；C1_* 为整数步；ibias 为安培
BASE = {
    "M1_L": 0.15,
    "M1_W": 0.55,
    "M1_M": 1,
    "M3_L": 0.15,
    "M3_W": 0.42,
    "M3_M": 1,
    "M5_L": 0.15,
    "M5_W": 0.42,
    "M5_M": 1,
    "M6_L": 0.15,
    "M6_W": 0.42,
    "M6_M": 1,
    "M7_L": 0.15,
    "M7_W": 0.55,
    "M7_M": 1,
    "M8_L": 0.15,
    "M8_W": 0.42,
    "M8_M": 1,
    "C1_L": 18,
    "C1_W": 18,
    "ibias": 3e-5,
}

SPACE = {
    # 重点调 M；L/W 小步；范围可按你的 config 再放宽
    "M3_M": (1, 36),
    "M7_M": (1, 44),
    "M6_L": (0.15, 0.55),
    "M5_W": (0.42, 2.2),
    "C1_W": (12, 36),
    "ibias": (1e-5, 8e-5),
}


# ============ 工具函数 ============
def clip(v, lo, hi):
    return max(lo, min(hi, v))


def sample_one(base: Dict[str, Any]) -> Dict[str, Any]:
    """随机采样一组参数（以 base 为底）"""
    p = dict(base)
    p["M3_M"] = random.randint(*SPACE["M3_M"])
    p["M4_M"] = p["M3_M"]  # 成对
    p["M7_M"] = random.randint(*SPACE["M7_M"])
    p["M8_M"] = p["M7_M"]

    p["M6_L"] = round(random.uniform(*SPACE["M6_L"]), 3)
    p["M5_W"] = round(random.uniform(*SPACE["M5_W"]), 2)
    p["C1_W"] = int(random.randint(*SPACE["C1_W"]))
    p["ibias"] = random.uniform(*SPACE["ibias"])
    return p


def merge_params(base: Dict[str, Any], delta: Dict[str, Any]) -> Dict[str, Any]:
    z = dict(base)
    z.update(delta)
    # 成对保证（若一边被改，另一边也跟）
    if "M3_M" in z and "M4_M" not in delta:
        z["M4_M"] = z["M3_M"]
    if "M7_M" in z and "M8_M" not in delta:
        z["M8_M"] = z["M7_M"]
    return z


def ok_gmid(op: Dict[str, Any]) -> bool:
    # 取关键管 gm/Id，若返回 None 视为不过滤
    def g(opd, key):
        x = opd.get(key, {})
        return x.get("gm/Id", None)

    def window(name):
        return GMID_WINDOW.get(name, (0.0, 1e9))

    gmids = {
        "m3": g(op, "m3"),
        "m4": g(op, "m4"),
        "m7": g(op, "m7"),
        "m8": g(op, "m8"),
    }
    for k, val in gmids.items():
        if val is None:
            continue
        lo, hi = window(k)
        if not (lo <= val <= hi):
            return False
    return True


def m6_sat_margin(op: Dict[str, Any]) -> float | None:
    m6 = op.get("m6", {})
    return m6.get("sat_margin", None)


def feasible(specs: Dict[str, Any], op: Dict[str, Any]) -> bool:
    pm_ok = specs.get("phase_margin", 0.0) >= HARD_CONSTRAINTS["pm_min"]
    sat = m6_sat_margin(op)
    sat_ok = (sat is not None) and (sat >= HARD_CONSTRAINTS["m6_sat_margin_min"])
    return pm_ok and sat_ok and ok_gmid(op)


def target_score(specs: Dict[str, Any]) -> float:
    """越大越好：把三项目标归一后加权求和"""
    dc = specs.get("dcgain_", 0.0) / (TARGETS["dcgain_"] + 1e-9)
    gbw = specs.get("gain_bandwidth_product_", 0.0) / (
        TARGETS["gain_bandwidth_product_"] + 1e-9
    )
    pm = specs.get("phase_margin", 0.0) / (TARGETS["phase_margin"] + 1e-9)
    # 保守些，先把 PM 放更高权重
    return 0.5 * pm + 0.3 * dc + 0.2 * gbw


# ============ 简易 Surrogate ============
# 用随机森林，稳定省心；你也可换成 GPR/LightGBM
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

FEATURES = ["M3_M", "M4_M", "M7_M", "M8_M", "M6_L", "M5_W", "C1_W", "ibias"]


def to_vec(p: Dict[str, Any]) -> List[float]:
    return [float(p.get(k, BASE[k])) for k in FEATURES]


@dataclass
class Surrogates:
    dc: Any
    gbw: Any
    pm: Any


def fit_surrogates(
    X: List[List[float]], Ydc: List[float], Ygbw: List[float], Ypm: List[float]
) -> Surrogates:
    dc = RandomForestRegressor(n_estimators=200, random_state=0)
    gbw = RandomForestRegressor(n_estimators=200, random_state=0)
    pm = RandomForestRegressor(n_estimators=200, random_state=0)
    dc.fit(X, Ydc)
    gbw.fit(X, Ygbw)
    pm.fit(X, Ypm)
    return Surrogates(dc=dc, gbw=gbw, pm=pm)


def predict_with_uncertainty(
    rf: RandomForestRegressor, X: List[List[float]]
) -> Tuple[List[float], List[float]]:
    # 平均值 + 基于树的方差近似
    all_preds = [tree.predict(X) for tree in rf.estimators_]
    means = [mean(vals) for vals in zip(*all_preds)]
    vars_ = []
    for i in range(len(X)):
        vs = [all_preds[t][i] for t in range(len(all_preds))]
        m = means[i]
        var = mean([(v - m) ** 2 for v in vs])
        vars_.append(var)
    return means, vars_


# ============ 主动学习循环 ============
def run_point(api: SimulationAPI, params: Dict[str, Any]) -> Dict[str, Any]:
    out = api.run(params=params)
    return out


def log_row(path: str, row: Dict[str, Any], header: List[str] | None):
    exists = os.path.exists(path)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=header or list(row.keys()))
        if not exists:
            w.writeheader()
        w.writerow(row)


def active_learning_demo(
    api: SimulationAPI,
    n_init: int = 24,
    rounds: int = 5,
    per_round: int = 12,
    log_csv: str = "al_log.csv",
):
    # Stage-0: random sweeping
    D_params: List[Dict[str, Any]] = []
    D_specs: List[Dict[str, Any]] = []
    D_op: List[Dict[str, Any]] = []

    print(f"[Stage-0] random sweeping: {n_init} points")
    for i in range(n_init):
        p = sample_one(BASE)
        res = run_point(api, p)
        D_params.append(p)
        D_specs.append(res["specs"])
        D_op.append(res["op_region"])
        row = {
            **p,
            **res["specs"],
            "m6_sat_margin": m6_sat_margin(res["op_region"]),
            "feasible": feasible(res["specs"], res["op_region"]),
        }
        log_row(log_csv, row, header=list(row.keys()))
        print(
            f"  #{i+1:03d} | score={target_score(res['specs']):.3f} | feasible={row['feasible']}"
        )

    # 主动学习
    for rd in range(rounds):
        print(f"[AL] round {rd+1}/{rounds}")
        # 训练 surrogate
        X = [to_vec(p) for p in D_params]
        Ydc = [s["dcgain_"] for s in D_specs]
        Ygbw = [s["gain_bandwidth_product_"] for s in D_specs]
        Ypm = [s["phase_margin"] for s in D_specs]
        sur = fit_surrogates(X, Ydc, Ygbw, Ypm)

        # 产生候选（random pool）并打分：目标 + 不确定性
        CANDS = 200
        pool: List[Dict[str, Any]] = [sample_one(BASE) for _ in range(CANDS)]
        Xp = [to_vec(p) for p in pool]

        dc_mean, dc_var = predict_with_uncertainty(sur.dc, Xp)
        gb_mean, gb_var = predict_with_uncertainty(sur.gbw, Xp)
        pm_mean, pm_var = predict_with_uncertainty(sur.pm, Xp)

        # 组合 acquisition：目标分数 + λ * 不确定性
        lam = 0.20  # 不确定性权重，可调
        scored = []
        for i, p in enumerate(pool):
            pred_specs = {
                "dcgain_": dc_mean[i],
                "gain_bandwidth_product_": gb_mean[i],
                "phase_margin": pm_mean[i],
            }
            s_obj = target_score(pred_specs)
            s_unc = math.sqrt(dc_var[i] + gb_var[i] + pm_var[i])
            score = s_obj + lam * s_unc
            scored.append((score, p, pred_specs))

        # 选出 per_round 个候选；但在提交仿真前做 **gm/Id 先验过滤**
        scored.sort(key=lambda x: x[0], reverse=True)
        taken = 0
        k = 0
        while taken < per_round and k < len(scored):
            _, p, _pred = scored[k]
            k += 1
            # 先仿真，拿到 OP 做硬约束/先验判断
            res = run_point(api, p)
            if not ok_gmid(res["op_region"]):
                continue
            if (
                m6_sat_margin(res["op_region"]) is None
                or m6_sat_margin(res["op_region"]) < 0.02
            ):
                # 太欠饱和，跳过；也可以保留为负例
                continue
            # 通过即纳入数据
            D_params.append(p)
            D_specs.append(res["specs"])
            D_op.append(res["op_region"])
            taken += 1
            row = {
                **p,
                **res["specs"],
                "m6_sat_margin": m6_sat_margin(res["op_region"]),
                "feasible": feasible(res["specs"], res["op_region"]),
            }
            log_row(log_csv, row, header=list(row.keys()))
            print(
                f"    + {taken:02d}/{per_round} | score={target_score(res['specs']):.3f} | feasible={row['feasible']}"
            )

        # 每轮结束给个汇总
        best_idx = max(range(len(D_specs)), key=lambda i: target_score(D_specs[i]))
        best_specs = D_specs[best_idx]
        best_params = D_params[best_idx]
        print(
            f"  ↪ best so far: score={target_score(best_specs):.3f} | "
            f"dc={best_specs['dcgain_']:.1f} dB, GBW={best_specs['gain_bandwidth_product_']:.0f} Hz, "
            f"PM={best_specs['phase_margin']:.1f}°, "
            f"M6_sat={m6_sat_margin(D_op[best_idx])}"
        )

    print("\n[Done] CSV saved to al_log.csv")


# ============ 直接运行 ============
if __name__ == "__main__":
    api = SimulationAPI()
    active_learning_demo(
        api,
        n_init=24,  # 随机起始预算
        rounds=5,  # AL 轮数
        per_round=12,  # 每轮仿真个数
        log_csv="al_log.csv",
    )

[Stage-0] random sweeping: 24 points
Simulation strategy: Use parameters from circuit.params section
Running single simulation...
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/temp/test_opamp_params.txt
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/opamp1_circuit.cir
Generated testbench circuit: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/opamp1_circuit.cir
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_op_region.spice
Generated circuit op region: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_op_region.spice
[✓] render done: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_circuit_params.txt
Generated circuit parameters: /home/G30079315/code/CircuitCollector/CircuitCollector/output/opamp/opamp1/temp_circuit_params.txt
[⚡] Running simulation: opamp1_circuit.cir
[✓] Simulat

KeyError: 'M4_M'

## Test redis

In [1]:
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

r.set("foo", "bar")
print(r.get("foo"))

bar
